In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, chi2

# === CARGA DE DATOS ===
train_data = pd.read_csv("../../Data/train_indexado.csv")

# === EXTRAER TEXTO Y ETIQUETAS MULTIETIQUETA ===
emotion_columns = train_data.columns.difference(['Text', 'ID'])
X_texts = train_data['Text']
Y = train_data[emotion_columns].values  # Matriz (n_samples, n_emotions)

# === TF-IDF ===
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    strip_accents='unicode',
    max_features=10000
)
X_tfidf = vectorizer.fit_transform(X_texts)

# === EVALUACIÓN PARA DIFERENTES K (excluyendo k=0) ===
k_values = [500, 1000, 2000, 3000, 5000]
scores = []

for k in k_values:
    chi2_selector = SelectKBest(chi2, k=k)
    X_selected = chi2_selector.fit_transform(X_tfidf, Y)

    clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
    f1_macro = cross_val_score(clf, X_selected, Y, cv=3, scoring='f1_macro').mean()
    print(f"F1_macro con {k} features: {f1_macro:.4f}")
    scores.append((k, f1_macro))

# === GRAFICAR ===
k_vals, f1_vals = zip(*scores)
plt.figure(figsize=(8, 5))
plt.plot(k_vals, f1_vals, marker='o', color='green')
plt.title('Evaluación del rendimiento con distintas k (SelectKBest + chi2)')
plt.xlabel('Número de Features seleccionadas (k)')
plt.ylabel('F1 Macro (Cross-Validated)')
plt.grid(True)
plt.tight_layout()
plt.savefig('../../Plots/chi2_selectkbest.png')
plt.close()

print("Gráfico guardado")

F1_macro con 500 features: 0.2130
F1_macro con 1000 features: 0.2138
F1_macro con 2000 features: 0.2128
F1_macro con 3000 features: 0.2124
F1_macro con 5000 features: 0.2096
Gráfico guardado


In [15]:
# === Selección de las 1000 mejores palabras ===
k_target = 1000
chi2_selector = SelectKBest(chi2, k=k_target)
X_1000 = chi2_selector.fit_transform(X_tfidf, Y)
top_1000_indices = chi2_selector.get_support(indices=True)

# === Cálculo del chi2 por emoción para cada palabra ===
chi2_scores_matrix = np.zeros((X_tfidf.shape[1], Y.shape[1]))

for i in range(Y.shape[1]):
    chi2_scores, _ = chi2(X_tfidf, Y[:, i])
    chi2_scores_matrix[:, i] = chi2_scores

# === Obtener nombres de las emociones ===
emotion_names = emotion_columns.tolist()

# === Obtener palabras y su emoción dominante ===
feature_names = vectorizer.get_feature_names_out()
selected_words = feature_names[top_1000_indices]
dominant_emotions = []

for idx in top_1000_indices:
    emotion_idx = np.argmax(chi2_scores_matrix[idx])
    dominant_emotions.append(emotion_names[emotion_idx])

# === Crear y guardar el DataFrame ===
df_word_emotion = pd.DataFrame({
    'Word': selected_words,
    'Emotion': dominant_emotions
})

output_path = "../../Data/Chi2/features_1000_with_emotion.csv"
df_word_emotion.to_csv(output_path, index=False)
print("Archivo guardado")

Archivo guardado
